# Ethical AI Bias + Explainability Analysis

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from fairlearn.metrics import demographic_parity_difference, equalized_odds_difference
from fairlearn.reductions import ExponentiatedGradient, DemographicParity
import shap

data = pd.read_csv('data.csv')

In [ ]:
X = data.drop('target', axis=1)
y = data['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

model = RandomForestClassifier()
model.fit(X_train, y_train)
preds = model.predict(X_test)

## Fairness BEFORE mitigation

In [ ]:
dp_before = demographic_parity_difference(y_test, preds)
eo_before = equalized_odds_difference(y_test, preds)
print("DP:", dp_before, "EO:", eo_before)

## Fairlearn Mitigation

In [ ]:
mitigator = ExponentiatedGradient(RandomForestClassifier(), constraints=DemographicParity())
mitigator.fit(X_train, y_train, sensitive_features=X_train['gender'])
preds_after = mitigator.predict(X_test)

## Fairness AFTER mitigation

In [ ]:
dp_after = demographic_parity_difference(y_test, preds_after)
eo_after = equalized_odds_difference(y_test, preds_after)
print("DP:", dp_after, "EO:", eo_after)

## SHAP Explainability

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)
shap.summary_plot(shap_values, X_test)